
Preprocess & fine-tune transformer....

Last Updated: July 21st, 2025

Daily Challenge : Preprocess & fine-tune transformer-based models


👩‍🏫 👩🏿‍🏫 What You’ll learn

In this daily challenge, you will learn how to preprocess and fine-tune transformer-based models, specifically BERT and XLM-RoBERTa, for text classification tasks. You will gain an understanding of:

    How tokenization works for these models.
    How to properly format input data.
    How to fine-tune transformer models for classification tasks.
    How to perform cross-validation using k-fold splitting.


🛠️ What you will create

By the end of this challenge, you will have a fine-tuned transformer model (BERT or XLM-RoBERTa) capable of classifying text into different categories. Additionally, you will structure the data for training, validate it using cross-validation, and understand how to optimize these models for better performance.


Dataset

You can find the dataset for this exercise here


Task


1. Understanding BERT and XLM-RoBERTa

Objective: Learn how transformer models work and their role in NLP tasks.

Instructions:

    Read through the descriptions of BERT and XLM-RoBERTa.
    Understand how these models process text using tokenization.
    Learn about different pre-trained versions of these models and their characteristics.

Functions to use:

    from transformers import BertTokenizer, XLMRobertaTokenizer


2. Tokenizing Text

Objective: Understand how to tokenize text using pre-trained tokenizers.

Instructions:

    Use the BertTokenizer and XLMRobertaTokenizer to convert sentences into tokenized input.
    Explore the different token types, such as input_ids, attention_mask, and labels.
    Experiment with single-sentence and two-sentence tokenization.

Functions to use:

    tokenizer.encode_plus()
    tokenizer.decode()


3. Preparing Input Data for the Model

Objective: Format input data correctly for transformer models.

Instructions:

    Ensure that input sentences are padded and possibly truncated to max_length.
    Understand and set special tokens such as <s> and </s>.
    Learn about attention_mask and how it helps the model ignore padding tokens.

Functions to use:

    tokenizer.encode_plus()
    tokenizer.special_tokens_map
    tokenizer.vocab_size


4. Loading and Exploring the Dataset

Objective: Load the dataset and explore its structure.

Instructions:

    Load the training and testing data from CSV files.
    Display the first few rows to understand its structure.
    Identify the columns needed for training the model.

Functions to use:

    pd.read_csv()
    df.head()
    df.shape


5. Creating Cross-Validation Folds

Objective: Implement k-fold cross-validation for training.

Instructions:

    Use StratifiedKFold to create 5 training-validation splits.
    Ensure that each fold maintains the same label distribution.
    Store the training and validation splits in separate lists.

Functions to use:

    from sklearn.model_selection import StratifiedKFold
    kf.split()
    StratifiedKFold(shuffle=True)


Cet exercice est un cas classique de classification de texte, plus spécifiquement de l'inférence de langage naturel (NLI - Natural Language Inference). L'objectif est de déterminer si une phrase "hypothèse" est une conséquence logique (entailment), une contradiction, ou neutre par rapport à une phrase "prémisse".

Le code suit un pipeline standard de projet NLP avec le modèle BERT et le framework PyTorch.

Étape 0 : Préparation de l'Environnement
C'est la toute première étape, qui consiste à s'assurer que toutes les bibliothèques nécessaires sont installées.
!pip -q install ... : Installe les bibliothèques principales :
transformers : Fournit l'accès à des modèles pré-entraînés comme BERT et à leurs tokenizers. C'est la bibliothèque clé de l'exercice.
datasets et evaluate : Outils de l'écosystème Hugging Face pour gérer les données et évaluer les modèles.
scikit-learn : Une bibliothèque classique de machine learning.
accelerate : Aide à optimiser l'entraînement sur différents types de matériel (CPU, GPU, TPU).
!pip install tensorflow : Bien que le code final utilise PyTorch, TensorFlow est installé. Dans ce notebook, il est utilisé au début pour préparer les données (tf.ragged.constant) avant que le reste du script ne soit adapté pour PyTorch.

Étape 1 : Chargement et Préparation des Données
Cette étape consiste à extraire les fichiers de données et à les charger en mémoire.
Décompression des archives : Le code décompresse une première archive ZIP (OUTER_ZIP), puis trouve les archives train.zip et test.zip à l'intérieur et les décompresse à leur tour. Cela permet d'accéder aux fichiers train.csv et test.csv.
Chargement avec Pandas : Les fichiers CSV sont chargés dans des DataFrames Pandas (train_df et test_df). Le DataFrame est une structure de données tabulaire très pratique pour manipuler les données.

Étape 2 : Exploration des Données (EDA - Exploratory Data Analysis)
Avant de construire un modèle, il est crucial de comprendre les données.
Aperçu des données : train_df.head() affiche les 5 premières lignes pour voir la structure (colonnes premise, hypothesis, label, etc.).
Distribution des classes : train_df['label'].value_counts() montre combien d'exemples il y a pour chaque catégorie (0: entailment, 1: neutral, 2: contradiction). C'est important pour vérifier si le jeu de données est équilibré.
Exemple concret : Le code sélectionne une paire prémisse/hypothèse au hasard et l'affiche avec son label textuel pour mieux comprendre la tâche.

Étape 3 : Prétraitement et Encodage des Données
Les modèles de deep learning comme BERT ne comprennent pas le texte brut. Il faut le transformer en nombres. C'est le rôle du tokenizer.
Charger le Tokenizer : BertTokenizer.from_pretrained('bert-base-multilingual-cased') charge un tokenizer pré-entraîné qui sait comment découper le texte en "tokens" (mots ou sous-mots) pour de nombreuses langues.
Créer la fonction d'encodage (bert_encode) : C'est une étape critique.
Elle prend les prémisses et les hypothèses.
Elle les combine en une seule chaîne de caractères, séparées par un token spécial [SEP]. C'est le format que BERT attend pour les tâches de classification de paires de phrases.
Elle utilise le tokenizer pour convertir ce texte en trois tenseurs principaux :
input_ids : L'identifiant numérique de chaque token.
attention_mask : Un masque binaire (0 ou 1) qui indique au modèle quels tokens sont réels et lesquels sont du "padding" (remplissage pour que toutes les séquences aient la même longueur).
token_type_ids : Indique à BERT quelle partie de l'entrée est la première phrase (la prémisse) et quelle partie est la seconde (l'hypothèse).

Étape 4 : Construction du Modèle
Ici, on définit l'architecture du modèle de classification.
Charger le modèle BERT pré-entraîné : BertModel.from_pretrained('bert-base-multilingual-cased') télécharge le corps principal du modèle BERT, déjà entraîné sur des milliards de phrases. On ne part pas de zéro, on fait du transfer learning.
Définir un classifieur personnalisé (BERTClassifier) : On crée une nouvelle classe en PyTorch (nn.Module).
Elle contient le modèle BERT comme une couche.
Elle ajoute une couche de Dropout pour éviter le surapprentissage (overfitting).
Elle ajoute une couche finale nn.Linear (couche "fully connected") qui prend la sortie de BERT et la projette sur 3 sorties (une pour chaque classe : entailment, neutral, contradiction).

Étape 5 : Préparation de l'Entraînement (PyTorch)
Cette partie configure tout ce qui est nécessaire pour lancer l'entraînement.
Conversion en Tenseurs PyTorch : Les données encodées (qui étaient des tenseurs TensorFlow ou des listes NumPy) sont converties en torch.tensor.
Création des Dataset et DataLoader :
TensorDataset regroupe les tenseurs d'entrée et les labels.
DataLoader est un utilitaire très puissant qui gère la création de mini-lots (batches) de données, le mélange des données à chaque époque (shuffle=True), etc. C'est essentiel pour un entraînement efficace.
Définition de l'Optimiseur et de la Fonction de Perte :
optimizer = torch.optim.Adam(...) : L'optimiseur (ici Adam) est l'algorithme qui met à jour les poids du modèle pour minimiser l'erreur.
criterion = nn.CrossEntropyLoss() : La fonction de perte (loss function) mesure l'écart entre les prédictions du modèle et les vrais labels. Pour la classification multi-classes, CrossEntropyLoss est le choix standard.
Configuration du matériel : Le code choisit d'utiliser le GPU (cuda) s'il est disponible, sinon le CPU.

Étape 6 : Entraînement du Modèle (La Boucle d'Entraînement)
C'est le cœur du processus d'apprentissage.
Boucle sur les époques (for epoch in range(epochs):) : Une époque est un passage complet sur l'ensemble des données d'entraînement.
Mode Entraînement (model.train()) : Active certaines couches comme le Dropout.
Boucle sur les batches (for batch in train_dataloader:) :
Récupère un lot de données.
Fait une passe avant (forward pass) : outputs = model(...) pour obtenir les prédictions.
Calcule la perte (loss) : loss = criterion(outputs, labels).
Fait une passe arrière (backward pass / backpropagation) : loss.backward() calcule comment chaque poids du modèle a contribué à l'erreur.
Met à jour les poids : optimizer.step().
Phase de Validation (model.eval()) : Après chaque époque, on évalue le modèle sur le jeu de données de validation.
model.eval() désactive le Dropout pour des prédictions stables.
with torch.no_grad(): désactive le calcul des gradients pour aller plus vite et économiser de la mémoire.
On calcule la perte et la précision sur les données de validation pour surveiller la performance du modèle sur des données qu'il n'a jamais vues pendant l'entraînement.

Étape 7 : Évaluation et Visualisation des Performances
Une fois l'entraînement terminé, on analyse les résultats.
matplotlib.pyplot : Le code trace des graphiques montrant l'évolution de la perte et de la précision (accuracy) pour les données d'entraînement et de validation au fil des époques.
Analyse des graphiques : Cela permet de voir si le modèle a bien appris et de détecter d'éventuels problèmes (ex: si la perte de validation remonte alors que celle d'entraînement continue de baisser, c'est un signe de surapprentissage).

Étape 8 : Inférence et Génération des Prédictions
L'objectif final est d'utiliser le modèle entraîné pour faire des prédictions sur de nouvelles données (le jeu de test).
Prétraitement des données de test : On applique exactement la même fonction bert_encode sur les données de test.
Création d'un DataLoader de test.
Mode Évaluation (model.eval()) : Très important pour s'assurer que les prédictions sont déterministes.
Boucle de prédiction : Le modèle parcourt les données de test et génère des prédictions. torch.max(outputs.data, 1) trouve la classe avec le score le plus élevé pour chaque exemple.
Création du fichier de soumission : Les prédictions sont ajoutées à un nouveau DataFrame et sauvegardées, généralement dans un fichier submission.csv, prêt à être soumis à une compétition (comme sur Kaggle).

Tokenization

    BERT utilise [CLS], [SEP] comme tokens spéciaux.

    XLM-RoBERTa utilise <s>, </s>.

In [1]:
import os, zipfile, glob, pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, BertTokenizer, XLMRobertaTokenizer, BertForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

In [13]:


bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

comprendre input_ids, attention_mask, token_type_ids.

In [14]:
sentence = "Transformers are amazing!"
encoded_bert = bert_tokenizer.encode_plus(
    sentence,
    add_special_tokens=True,
    max_length=32,
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

encoded_xlm = xlm_tokenizer.encode_plus(
    sentence,
    add_special_tokens=True,
    max_length=32,
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

# Affichage
print("BERT - input_ids:", encoded_bert['input_ids'])
print("BERT - attention_mask:", encoded_bert['attention_mask'])

print("XLM-RoBERTa - input_ids:", encoded_xlm['input_ids'])
print("XLM - attention_mask:", encoded_xlm['attention_mask'])

BERT - input_ids: tensor([[  101, 19081,  2024,  6429,   999,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0]])
BERT - attention_mask: tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])
XLM-RoBERTa - input_ids: tensor([[    0, 11062, 82772,     7,   621, 44613,    38,     2,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1]])
XLM - attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])


Préparer les données d'entrée

Tokenisation multiple 

In [15]:
text = "Deep Learning with transformers."
tokens = bert_tokenizer.encode_plus(
    text,
    add_special_tokens=True,
    padding='max_length',
    max_length=64,
    return_attention_mask=True,
    truncation=True,
    return_tensors='pt'
)

print("Special tokens:", bert_tokenizer.special_tokens_map)
print("Vocab size:", bert_tokenizer.vocab_size)

Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
Vocab size: 30522


In [16]:
# Chargement des CSV
TRAIN_CSV = "train.csv"
TEST_CSV  = "test.csv"

train_df = pd.read_csv(TRAIN_CSV)
print("\nTrain preview:\n", train_df.head())
print("\nLabel distribution:\n", train_df['label'].value_counts())

test_df = pd.read_csv(TEST_CSV) if os.path.exists(TEST_CSV) else None


Train preview:
            id                                            premise  \
0  5130fd2cb5  and these comments were considered in formulat...   
1  5b72532a0b  These are issues that we wrestle with in pract...   
2  3931fbe82a  Des petites choses comme celles-là font une di...   
3  5622f0c60b  you know they can't really defend themselves l...   
4  86aaa48b45  ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...   

                                          hypothesis lang_abv language  label  
0  The rules developed in the interim were put to...       en  English      0  
1  Practice groups are not permitted to work on t...       en  English      2  
2              J'essayais d'accomplir quelque chose.       fr   French      0  
3  They can't defend themselves because of their ...       en  English      0  
4    เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร       th     Thai      1  

Label distribution:
 label
0    4176
2    4064
1    3880
Name: count, dtype: int64


In [17]:


kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

folds = []
X_prem = train_df["premise"].values
X_hypo = train_df["hypothesis"].values
y = train_df["label"].values

for fold, (train_idx, val_idx) in enumerate(kf.split(X_prem, y)):
    print(f"Fold {fold+1} — Train: {len(train_idx)}, Val: {len(val_idx)}")
    folds.append((train_idx, val_idx))

Fold 1 — Train: 9696, Val: 2424
Fold 2 — Train: 9696, Val: 2424
Fold 3 — Train: 9696, Val: 2424
Fold 4 — Train: 9696, Val: 2424
Fold 5 — Train: 9696, Val: 2424


In [18]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

example = tokenizer.encode_plus(
    train_df.loc[0, "premise"],
    train_df.loc[0, "hypothesis"],
    add_special_tokens=True,
    padding='max_length',
    max_length=128,
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)

print(example)

{'input_ids': tensor([[ 101, 1998, 2122, 7928, 2020, 2641, 1999, 5675, 3436, 1996, 9455, 3513,
         1012,  102, 1996, 3513, 2764, 1999, 1996, 9455, 2020, 2404, 2362, 2007,
         2122, 7928, 1999, 2568, 1012,  102,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1,

In [2]:
import torch
print("GPU dispo :", torch.cuda.is_available())
print("Nom GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU dispo : True
Nom GPU : NVIDIA GeForce RTX 5060 Laptop GPU


c:\Users\mathi\Downloads\GenAI\GenAI_Bootcamp\nlp_env\lib\site-packages\torch\cuda\__init__.py:218: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [21]:
import logging
logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)
# 1. Parameters
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load Data
df = pd.read_csv("train.csv")
X = df[["premise", "hypothesis"]].values
y = df["label"].values

# 3. Dataset class
class NLIDataset(Dataset):
    def __init__(self, premise, hypothesis, labels, tokenizer, max_len):
        self.premise = premise
        self.hypothesis = hypothesis
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.premise)

    def __getitem__(self, idx):
        encodings = self.tokenizer.encode_plus(
            self.premise[idx],
            self.hypothesis[idx],
            add_special_tokens=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# 4. Training function
def train_epoch(model, loader, optimizer):
    model.train()
    losses = []
    for batch in tqdm(loader, desc="Training"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
    return np.mean(losses)

# 5. Evaluation function
def eval_model(model, loader):
    model.eval()
    predictions, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            labels = batch["labels"].cpu().numpy()
            predictions.extend(preds)
            targets.extend(labels)
    acc = accuracy_score(targets, predictions)
    f1 = f1_score(targets, predictions, average='weighted')
    return acc, f1

# 6. Tokenizer
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# 7. Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n🔥 Fold {fold+1}/5")

    train_dataset = NLIDataset(
        X[train_idx][:,0], X[train_idx][:,1], y[train_idx],
        tokenizer, MAX_LEN
    )
    val_dataset = NLIDataset(
        X[val_idx][:,0], X[val_idx][:,1], y[val_idx],
        tokenizer, MAX_LEN
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    for epoch in range(EPOCHS):
        print(f"\n📚 Epoch {epoch+1}/{EPOCHS}")
        train_loss = train_epoch(model, train_loader, optimizer)
        acc, f1 = eval_model(model, val_loader)
        print(f"Loss: {train_loss:.4f} | Val Accuracy: {acc:.4f} | F1: {f1:.4f}")

    # Save model for this fold
    output_dir = f"bert_fold_{fold+1}"
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)


🔥 Fold 1/5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



📚 Epoch 1/3


Training: 100%|██████████| 606/606 [23:39<00:00,  2.34s/it]


Loss: 1.0676 | Val Accuracy: 0.4695 | F1: 0.4435

📚 Epoch 2/3


...

Loss: 0.6689 | Val Accuracy: 0.5866 | F1: 0.5845

🔥 Fold 2/5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



📚 Epoch 1/3


Training:  11%|█         | 68/606 [02:38<20:51,  2.33s/it]


KeyboardInterrupt: 

In [ ]:
# 8. Test prediction using averaged ensemble
TEST_PATH = "data/Basics of BERT and XLM-RoBERTa - PyTorch - 2/test.csv"
if os.path.exists(TEST_PATH):
    print("\n🔮 Generating predictions on test set...")
    test_df = pd.read_csv(TEST_PATH)

    test_dataset = NLIDataset(
        test_df["premise"].values,
        test_df["hypothesis"].values,
        labels=[0] * len(test_df),  # Dummy labels
        tokenizer=tokenizer,
        max_len=MAX_LEN
    )
    test_loader = DataLoader(test_dataset, batch_size=32)

    all_logits = []

    for fold in range(5):
        model_path = f"bert_fold_{fold+1}"
        model = BertForSequenceClassification.from_pretrained(model_path).to(DEVICE)
        model.eval()

        fold_logits = []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits.cpu().numpy()
                fold_logits.append(logits)
        all_logits.append(np.vstack(fold_logits))

    # Moyenne des prédictions
    avg_logits = np.mean(all_logits, axis=0)
    predictions = np.argmax(avg_logits, axis=1)

    # Export CSV
    submission = pd.DataFrame({
        "id": test_df["id"],
        "predicted_label": predictions
    })
    submission.to_csv("submission.csv", index=False)
    print("✅ submission.csv sauvegardé avec succès.")
else:
    print("⚠️ Aucun fichier test.csv détecté — prédiction sautée.")

In [ ]:
# 1. Config
MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load training data
df = pd.read_csv("data/Basics of BERT and XLM-RoBERTa - PyTorch - 2/train.csv")
X = df[["premise", "hypothesis"]].values
y = df["label"].values

# 3. Dataset class
class NLIDataset(Dataset):
    def __init__(self, premise, hypothesis, labels, tokenizer, max_len):
        self.premise = premise
        self.hypothesis = hypothesis
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.premise)

    def __getitem__(self, idx):
        encodings = self.tokenizer.encode_plus(
            self.premise[idx],
            self.hypothesis[idx],
            add_special_tokens=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# 4. Training loop
def train_epoch(model, loader, optimizer):
    model.train()
    losses = []
    for batch in tqdm(loader, desc="Training"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
    return np.mean(losses)

# 5. Evaluation
def eval_model(model, loader):
    model.eval()
    predictions, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            labels = batch["labels"].cpu().numpy()
            predictions.extend(preds)
            targets.extend(labels)
    acc = accuracy_score(targets, predictions)
    f1 = f1_score(targets, predictions, average='weighted')
    return acc, f1

# 6. Tokenizer
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)

# 7. K-Fold training
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n🔥 Fold {fold+1}/5")

    train_dataset = NLIDataset(
        X[train_idx][:,0], X[train_idx][:,1], y[train_idx],
        tokenizer, MAX_LEN
    )
    val_dataset = NLIDataset(
        X[val_idx][:,0], X[val_idx][:,1], y[val_idx],
        tokenizer, MAX_LEN
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    for epoch in range(EPOCHS):
        print(f"\n📚 Epoch {epoch+1}/{EPOCHS}")
        train_loss = train_epoch(model, train_loader, optimizer)
        acc, f1 = eval_model(model, val_loader)
        print(f"Loss: {train_loss:.4f} | Val Accuracy: {acc:.4f} | F1: {f1:.4f}")

    output_dir = f"xlmr_fold_{fold+1}"
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

In [ ]:
# 8. Test set prediction with ensemble of XLM-R
TEST_PATH = "data/Basics of BERT and XLM-RoBERTa - PyTorch - 2/test.csv"
if os.path.exists(TEST_PATH):
    print("\n🔮 Generating predictions on test set...")
    test_df = pd.read_csv(TEST_PATH)

    test_dataset = NLIDataset(
        test_df["premise"].values,
        test_df["hypothesis"].values,
        labels=[0] * len(test_df),
        tokenizer=tokenizer,
        max_len=MAX_LEN
    )
    test_loader = DataLoader(test_dataset, batch_size=32)

    all_logits = []

    for fold in range(5):
        model_path = f"xlmr_fold_{fold+1}"
        model = XLMRobertaForSequenceClassification.from_pretrained(model_path).to(DEVICE)
        model.eval()

        fold_logits = []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits.cpu().numpy()
                fold_logits.append(logits)
        all_logits.append(np.vstack(fold_logits))

    avg_logits = np.mean(all_logits, axis=0)
    predictions = np.argmax(avg_logits, axis=1)

    submission = pd.DataFrame({
        "id": test_df["id"],
        "predicted_label": predictions
    })
    submission.to_csv("submission_xlmr.csv", index=False)
    print("✅ Fichier submission_xlmr.csv sauvegardé.")
else:
    print("⚠️ Aucun test.csv trouvé.")